# Cassava disease classifier — EfficientNetV2-S
## Drive is mounted first, and verified

## 1. Mount Drive — before anything else

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

assert os.path.ismount("/content/drive"), (
    "Drive did NOT mount. Anything saved to /content/drive would go to container "
    "disk and be lost when the runtime ends. Fix this before training.")
print("Drive mounted and verified")

## 2. Environment

In [ ]:
!nvidia-smi -L
import tensorflow as tf, keras
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE — switch runtime to T4")

## 3. Configuration

In [ ]:
import hashlib, json, shutil, datetime, gc
from dataclasses import dataclass, field, asdict, replace
from typing import List, Optional

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image as PILImage

AUTOTUNE = tf.data.AUTOTUNE
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

CROP = "cassava"

REGISTRY = [
    ("bacterial", "cbb",     "bacterial blight", "bacterial disease"),
    ("brown",     "cbs",     "brown spot",       "fungal disease"),
    ("mosaic",    "cmd",     "mosaic disease",   "viral disease"),
    ("healthy",   "healthy", "healthy",          "no problem detected"),
]

CLASSES = sorted(label for _, label, _, _ in REGISTRY)
SHORT = {label: disp for _, label, disp, _ in REGISTRY}
KIND = {label: kind for _, label, _, kind in REGISTRY}

def folder_to_label(name):
    low = name.lower()
    for keyword, label, _, _ in REGISTRY:
        if keyword in low:
            return label
    return None

@dataclass
class Config:
    data_root: str = ""
    crop: str = "cassava"
    img_size: int = 320
    batch_size: int = 24
    seed: int = 42

    backbone: str = "efficientnetv2s"
    head_dropout: float = 0.3

    head_epochs: int = 6
    finetune_epochs: int = 30
    head_lr: float = 1e-3
    finetune_lr: float = 1e-5
    unfreeze_last: int = 120
    freeze_batchnorm: bool = True
    label_smoothing: float = 0.05
    use_class_weights: bool = True
    early_stopping_patience: int = 8

    output_dir: str = "runs"
    class_names: Optional[List[str]] = field(default=None)

    @property
    def run_name(self):
        return f"{self.crop}_{self.backbone}"

CFG = Config(class_names=CLASSES)
keras.utils.set_random_seed(CFG.seed)

print(f"{len(CLASSES)} classes\n")
for c in CLASSES:
    print(f"  {SHORT[c]:<20} {KIND[c]}")

## 4. Download from Kaggle

In [ ]:
!pip install -q kaggle

from google.colab import files

print("Upload your kaggle.json")
print("(Kaggle → Settings → API → Create Legacy API Key)")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d gihozopatrick/cassava-dataset -p /content
!mkdir -p /content/data/cassava
!unzip -q -o /content/cassava-dataset.zip -d /content/data/cassava

import glob
for inner in glob.glob("/content/data/cassava/**/*.zip", recursive=True):
    print("unpacking nested:", os.path.basename(inner))
    !unzip -q -o "$inner" -d /content/data/cassava

found = None
for dirpath, dirnames, _ in os.walk("/content/data/cassava"):
    if all(d in dirnames for d in ("train", "valid", "test")):
        found = dirpath
        break
if found is None:
    raise FileNotFoundError("Could not find train/valid/test. Run !ls -R to inspect.")

CFG = replace(CFG, data_root=found)
print(f"\ndataset at: {CFG.data_root}\n")

for split in ("train", "valid", "test"):
    total = 0
    print(f"{split}/")
    for folder in sorted(os.listdir(os.path.join(CFG.data_root, split))):
        path = os.path.join(CFG.data_root, split, folder)
        if not os.path.isdir(path):
            continue
        label = folder_to_label(folder)
        n = len([f for f in os.listdir(path)
                 if os.path.splitext(f)[1].lower() in IMAGE_EXTS])
        total += n
        mark = SHORT[label] if label else "UNMAPPED"
        print(f"  {folder[:34]:<36} {n:>6}  -> {mark}")
    print(f"  {'TOTAL':<36} {total:>6}\n")

## 5. Resolutions and corrupt files

In [ ]:
from collections import Counter

print(f"{'split / class':<34}{'sizes':>28}{'bad':>6}")
print("-" * 70)
CORRUPT = set()
all_sizes = {}

for split in ("train", "valid", "test"):
    for folder in sorted(os.listdir(os.path.join(CFG.data_root, split))):
        path = os.path.join(CFG.data_root, split, folder)
        label = folder_to_label(folder) if os.path.isdir(path) else None
        if label is None:
            continue
        files = sorted(f for f in os.listdir(path)
                       if os.path.splitext(f)[1].lower() in IMAGE_EXTS)
        sizes, bad = Counter(), 0
        for f in files:
            fp = os.path.join(path, f)
            try:
                with PILImage.open(fp) as im:
                    im.verify()
                with PILImage.open(fp) as im:
                    sizes[im.size] += 1
            except Exception:
                CORRUPT.add(fp)
                bad += 1
        all_sizes[(split, label)] = sizes
        top = ", ".join(f"{w}x{h}" for w, h in [s for s, _ in sizes.most_common(2)])
        if len(sizes) > 2:
            top += f", +{len(sizes) - 2}"
        print(f"{split + ' / ' + SHORT[label]:<34}{top:>28}{bad:>6}")

print()
if CORRUPT:
    print(f"{len(CORRUPT)} unreadable files")
    for p in list(CORRUPT)[:6]:
        print(f"  {os.path.basename(p)}")
else:
    print("no unreadable files")

per_class = {}
for (split, label), sizes in all_sizes.items():
    per_class.setdefault(label, Counter()).update(sizes)
signatures = {label: set(s) for label, s in per_class.items()}
print()
if len(set(map(frozenset, signatures.values()))) == 1:
    print("every class shares the same resolution profile — no geometry shortcut")
else:
    print("WARNING: resolution profiles differ between classes. Check whether a "
          "model could separate a class on image geometry alone before training.")
    for label, s in signatures.items():
        print(f"  {SHORT[label]:<20} {sorted(s)[:4]}")

## 6. Build the splits

In [ ]:
def file_hash(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def phash(path, size=8):
    try:
        with PILImage.open(path) as im:
            arr = np.asarray(im.convert("L").resize((size, size), PILImage.BILINEAR),
                             dtype="float32")
        return "".join("1" if b else "0" for b in (arr > arr.mean()).flatten())
    except Exception:
        return None

def index_split(root, split):
    rows = []
    base = os.path.join(root, split)
    for folder in sorted(os.listdir(base)):
        path = os.path.join(base, folder)
        if not os.path.isdir(path):
            continue
        label = folder_to_label(folder)
        if label is None:
            print(f"[data] skipping unmapped folder: {folder}")
            continue
        for fname in sorted(os.listdir(path)):
            if os.path.splitext(fname)[1].lower() in IMAGE_EXTS:
                rows.append((os.path.join(path, fname), label))
    return pd.DataFrame(rows, columns=["filepath", "label"])

parts = {}
for split in ("train", "valid", "test"):
    d = index_split(CFG.data_root, split)
    before = len(d)
    d = d[~d["filepath"].isin(CORRUPT)].reset_index(drop=True)
    if before != len(d):
        print(f"[data] {split}: {before - len(d)} unreadable files removed")

    d["hash"] = d["filepath"].map(file_hash)
    d["phash"] = d["filepath"].map(phash)
    n_bad = d["phash"].isna().sum()
    if n_bad:
        print(f"[data] {split}: {n_bad} further files failed to decode — removed")
        d = d[d["phash"].notna()].reset_index(drop=True)

    before = len(d)
    d = d.drop_duplicates(subset="hash", keep="first").reset_index(drop=True)
    d = d.drop_duplicates(subset="phash", keep="first").reset_index(drop=True)
    print(f"[data] {split}: {before - len(d)} internal duplicates removed, {len(d)} kept")
    parts[split] = d

eval_keys = set(parts["valid"]["phash"]) | set(parts["test"]["phash"])
overlap = parts["train"]["phash"].isin(eval_keys)
if overlap.any():
    print(f"\n[data] CONTAMINATION: {overlap.sum()} train images "
          f"({100 * overlap.mean():.1f}%) also appear in valid/test — "
          f"dropping them from TRAIN so evaluation stays clean")
    parts["train"] = parts["train"][~overlap].reset_index(drop=True)
else:
    print("\n[data] no train/eval overlap found")

vt = set(parts["valid"]["phash"]) & set(parts["test"]["phash"])
if vt:
    print(f"[data] {len(vt)} images shared between valid and test — dropping from VAL")
    parts["valid"] = parts["valid"][~parts["valid"]["phash"].isin(vt)].reset_index(drop=True)

TRAIN_DF, VAL_DF, TEST_DF = parts["train"], parts["valid"], parts["test"]

print()
for c in CLASSES:
    print(f"  {SHORT[c]:<20} "
          f"train {(TRAIN_DF['label'] == c).sum():>5}  "
          f"val {(VAL_DF['label'] == c).sum():>4}  "
          f"test {(TEST_DF['label'] == c).sum():>4}")

counts = TRAIN_DF["label"].value_counts()
print(f"\nimbalance ratio: {counts.max() / counts.min():.1f}:1")
print(f"smallest test class: {TEST_DF['label'].value_counts().min()} images")

## 7. Validate with TensorFlow's decoder

In [ ]:
def tf_readable(path):
    try:
        img = tf.io.decode_image(tf.io.read_file(path), channels=3,
                                 expand_animations=False)
        _ = img.shape
        return True
    except Exception:
        return False

bad = set()
for name, part in (("train", TRAIN_DF), ("valid", VAL_DF), ("test", TEST_DF)):
    print(f"checking {name} ({len(part)} files)...")
    for p in part["filepath"]:
        if not tf_readable(p):
            bad.add(p)

print(f"\n[data] {len(bad)} files rejected by TensorFlow's decoder")
for p in list(bad)[:6]:
    print(f"  {os.path.basename(p)}")

if bad:
    TRAIN_DF = TRAIN_DF[~TRAIN_DF["filepath"].isin(bad)].reset_index(drop=True)
    VAL_DF = VAL_DF[~VAL_DF["filepath"].isin(bad)].reset_index(drop=True)
    TEST_DF = TEST_DF[~TEST_DF["filepath"].isin(bad)].reset_index(drop=True)

print(f"\n[data] train {len(TRAIN_DF)}  val {len(VAL_DF)}  test {len(TEST_DF)}")

## 8. Data pipeline

In [ ]:
def augment(image, seed_pair):
    image = tf.image.stateless_random_flip_left_right(image, seed_pair)
    image = tf.image.stateless_random_flip_up_down(image, seed_pair)
    image = tf.image.stateless_random_brightness(image, 0.20, seed_pair)
    image = tf.image.stateless_random_contrast(image, 0.80, 1.20, seed_pair)
    image = tf.image.stateless_random_saturation(image, 0.90, 1.10, seed_pair)
    image = tf.image.stateless_random_hue(image, 0.02, seed_pair)
    return tf.clip_by_value(image, 0.0, 255.0)

def make_dataset(df, classes, cfg, training, seed=None):
    seed = cfg.seed if seed is None else seed
    idx = {c: i for i, c in enumerate(classes)}
    paths = df["filepath"].to_numpy()
    labels = df["label"].map(idx).to_numpy().astype("int32")

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

    def load(path, label):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img = tf.image.resize(img, (cfg.img_size, cfg.img_size), method="bilinear")
        return tf.cast(img, tf.float32), tf.one_hot(label, len(classes))

    ds = ds.map(load, num_parallel_calls=AUTOTUNE)

    if training:
        ds = tf.data.Dataset.zip((ds, tf.data.Dataset.counter()))
        ds = ds.map(lambda xy, c: (augment(xy[0], tf.stack([c, seed])), xy[1]),
                    num_parallel_calls=AUTOTUNE)

    return ds.batch(cfg.batch_size).prefetch(AUTOTUNE)

def class_weights(df, classes):
    counts = df["label"].value_counts()
    freqs = np.array([counts.get(c, 0) for c in classes], dtype="float64")
    w = freqs.sum() / (len(classes) * freqs)
    print("[data] class weights: " +
          ", ".join(f"{SHORT[c]}={v:.2f}" for c, v in zip(classes, w)))
    return {i: float(v) for i, v in enumerate(w)}

print("data pipeline defined")

## 9. Model

In [ ]:
BACKBONES = {
    "efficientnetv2s": ("EfficientNetV2S",  None),
    "convnext_tiny":   ("ConvNeXtTiny",     None),
    "mobilenetv3":     ("MobileNetV3Large", None),
    "densenet121":     ("DenseNet121",      "densenet"),
    "resnet50":        ("ResNet50",         "resnet50"),
}

@keras.saving.register_keras_serializable(package="cropdx")
class Preprocess(layers.Layer):

    def __init__(self, module=None, **kwargs):
        super().__init__(**kwargs)
        self.module = module

    def call(self, inputs):
        if self.module is None:
            return inputs
        return getattr(keras.applications, self.module).preprocess_input(inputs)

    def get_config(self):
        c = super().get_config(); c.update({"module": self.module}); return c

def build_model(cfg, n_classes):
    cls_name, module = BACKBONES[cfg.backbone]
    inputs = layers.Input((cfg.img_size, cfg.img_size, 3), name="image")
    x = Preprocess(module, name="preprocess")(inputs)
    backbone = getattr(keras.applications, cls_name)(
        include_top=False, weights="imagenet",
        input_shape=(cfg.img_size, cfg.img_size, 3))
    backbone.trainable = False
    f = backbone(x)
    p = layers.GlobalAveragePooling2D(name="gap")(f)
    p = layers.Dropout(cfg.head_dropout, name="head_dropout")(p)
    out = layers.Dense(n_classes, activation="softmax", dtype="float32",
                       name="predictions")(p)
    return keras.Model(inputs, out, name=cfg.run_name)

def get_backbone(model, cfg):
    for cand in (cfg.backbone, BACKBONES[cfg.backbone][0].lower()):
        try:
            return model.get_layer(cand)
        except ValueError:
            continue
    for layer in model.layers:
        if isinstance(layer, keras.Model) and len(layer.layers) > 10:
            return layer
    raise ValueError("backbone not found")

def unfreeze_top(model, cfg):
    backbone = get_backbone(model, cfg)
    backbone.trainable = True
    cutoff = len(backbone.layers) - cfg.unfreeze_last
    n = 0
    for i, l in enumerate(backbone.layers):
        if i < cutoff:
            l.trainable = False
        elif cfg.freeze_batchnorm and isinstance(l, layers.BatchNormalization):
            l.trainable = False
        else:
            l.trainable = True; n += 1
    print(f"[model] fine-tuning {n}/{len(backbone.layers)} backbone layers")

print("model functions defined")

## 10. Training and evaluation

In [ ]:
def train(cfg, train_df, val_df, test_df, verbose=1):
    run_dir = os.path.join(cfg.output_dir, cfg.run_name)
    os.makedirs(run_dir, exist_ok=True)
    keras.utils.set_random_seed(cfg.seed)

    for name, part in (("train", train_df), ("val", val_df), ("test", test_df)):
        part.to_csv(os.path.join(run_dir, f"split_{name}.csv"), index=False)
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump({**asdict(cfg), "run_name": cfg.run_name,
                   "display_names": SHORT, "kinds": KIND,
                   "source": "kaggle gihozopatrick/cassava-dataset, provided split"},
                  f, indent=2)

    train_ds = make_dataset(train_df, cfg.class_names, cfg, training=True)
    val_ds = make_dataset(val_df, cfg.class_names, cfg, training=False)

    weights = class_weights(train_df, cfg.class_names) if cfg.use_class_weights else None
    loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=cfg.label_smoothing)

    model = build_model(cfg, len(cfg.class_names))
    print(f"[train] {cfg.run_name} at {cfg.img_size}px: {model.count_params():,} params")

    callbacks = [
        keras.callbacks.ModelCheckpoint(os.path.join(run_dir, "best.keras"),
                                        monitor="val_loss", save_best_only=True),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=cfg.early_stopping_patience,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2,
                                          min_lr=1e-7, verbose=1),
        keras.callbacks.CSVLogger(os.path.join(run_dir, "history.csv"), append=True),
    ]

    print("\n[train] phase 1/2 — frozen backbone")
    model.compile(optimizer=keras.optimizers.Adam(cfg.head_lr), loss=loss_fn,
                  metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=cfg.head_epochs,
                   class_weight=weights, callbacks=callbacks, verbose=verbose)

    print("\n[train] phase 2/2 — fine-tuning")
    unfreeze_top(model, cfg)
    model.compile(optimizer=keras.optimizers.Adam(cfg.finetune_lr), loss=loss_fn,
                  metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds,
                   epochs=cfg.head_epochs + cfg.finetune_epochs,
                   initial_epoch=len(h1.history["loss"]),
                   class_weight=weights, callbacks=callbacks, verbose=verbose)

    model.save(os.path.join(run_dir, "final.keras"))
    history = {k: [float(v) for v in h1.history.get(k, []) + h2.history.get(k, [])]
               for k in set(h1.history) | set(h2.history)}
    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)
    print(f"\n[train] best val_accuracy={max(history['val_accuracy']):.4f}")
    return model, history, run_dir

def evaluate(model, df, name, cfg=None, show=True):
    cfg = cfg or CFG
    ds = make_dataset(df, cfg.class_names, cfg, training=False)
    probs = model.predict(ds, verbose=1)
    y_pred = probs.argmax(axis=1)
    y_true = df["label"].map({c: i for i, c in enumerate(cfg.class_names)}).to_numpy()

    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro")
    conf = probs.max(axis=1)
    ok = y_pred == y_true

    print(f"\n===== {name} — {len(df)} images =====")
    print(classification_report(y_true, y_pred, labels=list(range(len(cfg.class_names))),
                                target_names=[SHORT[c] for c in cfg.class_names],
                                digits=3, zero_division=0))
    print(f"accuracy : {acc:.4f}    macro F1 : {macro:.4f}")
    print(f"confidence — correct {conf[ok].mean():.4f}, wrong {conf[~ok].mean():.4f}, "
          f"separation {conf[ok].mean() - conf[~ok].mean():+.4f}")

    if show:
        cmx = confusion_matrix(y_true, y_pred, labels=list(range(len(cfg.class_names))))
        plt.figure(figsize=(7, 5.8))
        sns.heatmap(cmx, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=[SHORT[c] for c in cfg.class_names],
                    yticklabels=[SHORT[c] for c in cfg.class_names])
        plt.xlabel("Predicted"); plt.ylabel("True")
        plt.title(f"{name} — acc {acc:.3f}, macro-F1 {macro:.3f}")
        plt.xticks(rotation=25, ha="right"); plt.yticks(rotation=0)
        plt.tight_layout(); plt.show()

    return {"name": name, "n": int(len(df)), "accuracy": float(acc),
            "macro_f1": float(macro), "conf_correct": float(conf[ok].mean()),
            "conf_wrong": float(conf[~ok].mean()), "probs": probs,
            "y_true": y_true, "y_pred": y_pred}

print("training and evaluation defined")

## 11. Smoke test — about two minutes

In [ ]:
smoke = replace(CFG, head_epochs=1, finetune_epochs=1, img_size=128,
                output_dir="runs_smoke")
_ = train(smoke,
          TRAIN_DF.sample(min(500, len(TRAIN_DF)), random_state=0),
          VAL_DF.sample(min(150, len(VAL_DF)), random_state=0),
          TEST_DF.sample(min(150, len(TEST_DF)), random_state=0))
print("\npipeline OK")

## 12. Train

In [ ]:
model, history, run_dir = train(CFG, TRAIN_DF, VAL_DF, TEST_DF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["accuracy"], label="train")
axes[0].plot(history["val_accuracy"], label="val")
axes[0].set_title("accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history["loss"], label="train")
axes[1].plot(history["val_loss"], label="val")
axes[1].set_title("loss"); axes[1].set_xlabel("epoch"); axes[1].legend()
for ax in axes:
    ax.axvline(CFG.head_epochs - 0.5, color="grey", ls="--", lw=1)
plt.tight_layout(); plt.show()
print("dashed line = start of fine-tuning")

## 13. Save immediately — before anything else

In [ ]:
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
dest = f"/content/drive/MyDrive/cassava_v2/run_{stamp}"
os.makedirs("/content/drive/MyDrive/cassava_v2", exist_ok=True)
shutil.copytree("runs", dest)

saved = os.path.join(dest, CFG.run_name, "best.keras")
assert os.path.exists(saved), f"{saved} is missing — the save did not work"
assert os.path.getsize(saved) > 1_000_000, "saved model is suspiciously small"
print(f"saved and verified: {saved}")
!du -sh "$dest"

## 14. Evaluate on the held-out test split

In [ ]:
test = evaluate(model, TEST_DF, "cassava — held-out test")

os.makedirs("results", exist_ok=True)
with open("results/cassava_test.json", "w") as f:
    json.dump({**{k: test[k] for k in
                  ("name", "n", "accuracy", "macro_f1", "conf_correct", "conf_wrong")},
               "crop": "cassava", "classes": CLASSES, "kinds": KIND,
               "backbone": CFG.backbone, "img_size": CFG.img_size}, f, indent=2)
print("wrote results/cassava_test.json")

## 15. Calibrate an abstain threshold

In [ ]:
val = evaluate(model, VAL_DF, "cassava — validation (calibration)", show=False)
conf = val["probs"].max(axis=1)
ok = val["y_pred"] == val["y_true"]

lines = ["", "ABSTAIN THRESHOLD SWEEP — cassava (validation)", "",
         f"mean confidence — correct {conf[ok].mean():.4f}, wrong {conf[~ok].mean():.4f}",
         f"separation {conf[ok].mean() - conf[~ok].mean():+.4f}", "",
         f"{'threshold':>10}{'refer %':>10}{'accuracy':>11}{'errors left':>13}", "-" * 44]

sweep = []
for t in [0.0, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]:
    keep = conf >= t
    if keep.sum() == 0:
        continue
    sweep.append({"threshold": t, "refer_pct": float(100 * (1 - keep.mean())),
                  "accuracy": float(ok[keep].mean()), "errors": int((~ok[keep]).sum())})
    lines.append(f"{t:>10.2f}{100 * (1 - keep.mean()):>9.1f}%"
                 f"{ok[keep].mean():>11.4f}{int((~ok[keep]).sum()):>13d}")

table = "\n".join(lines + [""])
print(table)
with open("results/cassava_calibration.json", "w") as f:
    json.dump({"crop": "cassava",
               "separation": float(conf[ok].mean() - conf[~ok].mean()),
               "sweep": sweep}, f, indent=2)
print("Pick the point where the accuracy gain stops paying for the referral rate.")

## 16. Where is the model looking?

In [ ]:
def split_model(m):
    backbone = next(l for l in m.layers
                    if isinstance(l, keras.Model) and len(l.layers) > 10)
    conv_name = None
    for layer in reversed(backbone.layers):
        try:
            shape = layer.output.shape
        except AttributeError:
            continue
        if len(shape) == 4:
            conv_name = layer.name; break
    idx = m.layers.index(backbone)
    return (m.layers[1:idx],
            keras.Model(backbone.input, backbone.get_layer(conv_name).output),
            m.layers[idx + 1:])

def gradcam(split, batch, class_index=None):
    pre, conv_sub, post = split
    x = batch
    for l in pre:
        x = l(x)
    with tf.GradientTape() as tape:
        conv_out = conv_sub(x, training=False)
        tape.watch(conv_out)
        y = conv_out
        for l in post:
            y = l(y, training=False)
        if class_index is None:
            class_index = int(tf.argmax(y[0]))
        score = y[:, class_index]
    grads = tape.gradient(score, conv_out)
    w = tf.reduce_mean(grads, axis=(0, 1, 2))
    h = tf.maximum(tf.reduce_sum(conv_out[0] * w, axis=-1), 0)
    return (h / (tf.reduce_max(h) + 1e-8)).numpy(), class_index

def overlay(pil, heat, alpha=0.5):
    col = cm.jet(heat)[..., :3]
    col = PILImage.fromarray(np.uint8(col * 255)).resize(pil.size, PILImage.BILINEAR)
    return PILImage.blend(pil, col, alpha)

SPLIT = split_model(model)

def show_attention(df, title, n=6, want_correct=None, y_pred=None, seed=1):
    work = df.reset_index(drop=True)
    if want_correct is not None and y_pred is not None:
        truth = work["label"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
        mask = (y_pred == truth) if want_correct else (y_pred != truth)
        work = work[mask].reset_index(drop=True)
    if len(work) == 0:
        print(f"{title}: nothing to show"); return

    rng = np.random.default_rng(seed)
    picks = rng.choice(len(work), size=min(n, len(work)), replace=False)

    fig, axes = plt.subplots(2, len(picks), figsize=(3.0 * len(picks), 6.4))
    if len(picks) == 1:
        axes = axes.reshape(2, 1)
    for col, i in enumerate(picks):
        row = work.iloc[i]
        pil = PILImage.open(row["filepath"]).convert("RGB").resize(
            (CFG.img_size, CFG.img_size))
        batch = np.expand_dims(np.asarray(pil, dtype="float32"), 0)
        probs = model.predict(batch, verbose=0)[0]
        pred = int(probs.argmax())
        heat, _ = gradcam(SPLIT, batch, class_index=pred)
        good = CLASSES[pred] == row["label"]

        axes[0, col].imshow(pil)
        axes[0, col].set_title(f"true: {SHORT[row['label']]}", fontsize=8.5)
        axes[1, col].imshow(overlay(pil, heat))
        axes[1, col].set_title(f"{'OK  ' if good else 'WRONG  '}"
                               f"{SHORT[CLASSES[pred]]}\n{probs[pred]:.2f}",
                               fontsize=8.5, color="green" if good else "firebrick")
        for r in range(2):
            axes[r, col].axis("off")
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()

show_attention(TEST_DF, "cassava — correctly classified",
               want_correct=True, y_pred=test["y_pred"])
show_attention(TEST_DF, "cassava — misclassified: what did it look at instead?",
               want_correct=False, y_pred=test["y_pred"])

## 17. Background dependence

In [ ]:
def leaf_mask(image, sat_thresh=0.20, val_thresh=0.12, smooth=9):
    hsv = tf.image.rgb_to_hsv(image / 255.0)
    sat, val = hsv[..., 1], hsv[..., 2]
    mask = tf.cast((sat > sat_thresh) & (val > val_thresh), tf.float32)[..., None]
    mask = tf.nn.avg_pool2d(mask[None], ksize=smooth, strides=1, padding="SAME")[0]
    return tf.cast(mask > 0.4, tf.float32)

def make_background(shape, seed_pair):
    h, w = shape[0], shape[1]
    solid = tf.random.stateless_uniform([1, 1, 3], seed_pair, 0, 255) * tf.ones([h, w, 3])
    noise = tf.random.stateless_uniform([h, w, 3], seed_pair + 1, 0, 255)
    coarse = tf.image.resize(
        tf.random.stateless_uniform([8, 8, 3], seed_pair + 2, 0, 255), (h, w), method="bicubic")
    which = tf.random.stateless_uniform([], seed_pair + 3, 0, 3, dtype=tf.int32)
    return tf.clip_by_value(
        tf.switch_case(which, [lambda: solid, lambda: noise, lambda: coarse]), 0.0, 255.0)

def make_dataset_randbg(df, classes, cfg, seed=7):
    idx = {c: i for i, c in enumerate(classes)}
    paths = df["filepath"].to_numpy()
    labels = df["label"].map(idx).to_numpy().astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load(path, label):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img = tf.image.resize(img, (cfg.img_size, cfg.img_size), method="bilinear")
        return tf.cast(img, tf.float32), tf.one_hot(label, len(classes))

    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    ds = tf.data.Dataset.zip((ds, tf.data.Dataset.counter()))

    def swap(xy, c):
        img, lab = xy
        pair = tf.stack([c, seed])
        m = leaf_mask(img)
        return img * m + make_background(tf.shape(img), pair) * (1.0 - m), lab

    ds = ds.map(swap, num_parallel_calls=AUTOTUNE)
    return ds.batch(cfg.batch_size).prefetch(AUTOTUNE)

rng = np.random.default_rng(3)
picks = rng.choice(len(TEST_DF), size=5, replace=False)
fig, axes = plt.subplots(3, 5, figsize=(13, 7.6))
for col, i in enumerate(picks):
    row = TEST_DF.iloc[i]
    pil = PILImage.open(row["filepath"]).convert("RGB").resize((CFG.img_size, CFG.img_size))
    arr = tf.constant(np.asarray(pil, dtype="float32"))
    m = leaf_mask(arr)
    swapped = arr * m + make_background(tf.shape(arr), tf.constant([col, 7], tf.int32)) * (1.0 - m)
    axes[0, col].imshow(pil); axes[0, col].set_title(SHORT[row["label"]], fontsize=8)
    axes[1, col].imshow(m.numpy()[..., 0], cmap="gray")
    axes[1, col].set_title(f"kept {100 * m.numpy().mean():.0f}% of frame", fontsize=8)
    axes[2, col].imshow(swapped.numpy().astype("uint8"))
    axes[2, col].set_title("background replaced", fontsize=8)
    for r in range(3):
        axes[r, col].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
clean_probs = model.predict(make_dataset(TEST_DF, CLASSES, CFG, training=False), verbose=1)
swap_probs = model.predict(make_dataset_randbg(TEST_DF, CLASSES, CFG), verbose=1)

y_true = TEST_DF["label"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
clean_acc = accuracy_score(y_true, clean_probs.argmax(axis=1))
swap_acc = accuracy_score(y_true, swap_probs.argmax(axis=1))

lines = ["", "BACKGROUND DEPENDENCE — cassava", "",
         f"  untouched            {clean_acc:.4f}",
         f"  background replaced  {swap_acc:.4f}",
         f"  drop: {100 * (clean_acc - swap_acc):.1f} accuracy points", "",
         f"{'class':<20}{'clean':>10}{'swapped':>10}{'drop':>8}", "-" * 48]
for i, c in enumerate(CLASSES):
    m = y_true == i
    a1 = accuracy_score(y_true[m], clean_probs.argmax(axis=1)[m])
    a2 = accuracy_score(y_true[m], swap_probs.argmax(axis=1)[m])
    lines.append(f"{SHORT[c]:<20}{a1:>10.4f}{a2:>10.4f}{100 * (a1 - a2):>7.1f}")

lines += ["", "The same diagnostic across this project:",
          "  PlantVillage tomato model:      29.8 point drop",
          "  curated maize collection:       32.4 point drop",
          "  CCMT tomato (field-captured):   14.5 point drop",
          "  CCMT maize (field-captured):     4.6 point drop", "",
          "Caveat: the mask is colour-based, so brown soil next to a tan lesion",
          "is imperfectly separated. Treat the figure as indicative.", ""]

table = "\n".join(lines)
print(table)
with open("results/cassava_background.json", "w") as f:
    json.dump({"crop": "cassava",
               "clean_accuracy": float(clean_acc), "swapped_accuracy": float(swap_acc),
               "drop_points": float(100 * (clean_acc - swap_acc))}, f, indent=2)
print("wrote results/cassava_background.json")

## 18. Save the results

In [ ]:
shutil.copytree("results", dest + "_results", dirs_exist_ok=True)
assert os.path.exists(os.path.join(dest + "_results", "cassava_test.json"))
print("results saved and verified:", dest + "_results")
!ls -lh "$dest"_results

## What to report